# Banc d'essai : Albert classe-t-il les articles comme nous ?

À partir du Sprint 3, un modèle d'Albert classera seul des milliers d'articles. Ce carnet vérifie, sur 20 articles dont on connaît la bonne réponse (`tests/reference_20.csv`), **quel modèle étiquette le mieux**.

1. Il affiche vos **quotas** Albert et la **liste des modèles**.
2. Il lit les 20 articles **une seule fois**.
3. Il fait classer ces articles par **chaque modèle de la liste** (tri, puis fiche complète).
4. Il affiche un **tableau comparatif** : pour chaque étiquette, la part d'articles où le modèle donne la même réponse que la référence.

**Une seule fois** : icône 🔑 à gauche > secret `ALBERT_API_KEY` > « Accès au notebook » activé.
Puis **Exécution > Tout exécuter**. Comptez 5 minutes par modèle.

In [ ]:
!pip install -q openai anthropic trafilatura pandas

In [ ]:
import json, re, time, logging
import pandas as pd
import requests
import trafilatura
from openai import OpenAI
from google.colab import userdata

BRANCHE = "v2-sprint1"
DEPOT = f"https://raw.githubusercontent.com/GabrielPlassat/tableaubord_innov/{BRANCHE}"
ADRESSE_ALBERT = "https://albert.api.etalab.gouv.fr/v1"
CLE = userdata.get("ALBERT_API_KEY")
client = OpenAI(base_url=ADRESSE_ALBERT, api_key=CLE)
logging.getLogger("trafilatura").setLevel(logging.CRITICAL)
print("Connexion prête.")

## 1. Quotas et modèles disponibles
Copiez la sortie de la cellule des quotas dans `docs/verifications.md` (ou envoyez-la à Claude).

In [ ]:
r = requests.get(f"{ADRESSE_ALBERT}/me/info", headers={"Authorization": f"Bearer {CLE}"}, timeout=30)
print("Statut :", r.status_code)
info = r.json() if r.ok else {"erreur": r.text[:500]}
print(json.dumps(info.get("limits", info), indent=2, ensure_ascii=False)[:4000])

In [ ]:
modeles = pd.DataFrame([{"id": m.model_dump().get("id"), "type": m.model_dump().get("type", "")}
                        for m in client.models.list().data])
display(modeles)

## 2. Les modèles à comparer

Liste modifiable. Par défaut : les modèles de génération de texte généralistes (on écarte les modèles spécialisés dans le code ou l'OCR).

In [ ]:
MODELES_A_COMPARER = [
    "mistral-small-3-2-24b-instruct-2506",   # Mistral (France), sans phase de réflexion
    "gemma-4-31b-it",                        # Google, modèle ouvert
    "deepseek-v4-flash-0731",                # DeepSeek (Chine) : à soumettre au test d'omission
    "gpt-oss-120b",                          # OpenAI, modèle ouvert, avec phase de réflexion
    "ministral-3-8b-instruct-2512",          # Mistral, petit modèle : le moins gourmand en quota
]
VERSION_CONSIGNE = "v1"   # "v1" ou "v2" (v2 : définitions resserrées et exemples)
disponibles = set(modeles["id"])
MODELES_A_COMPARER = [m for m in MODELES_A_COMPARER if m in disponibles]

# Modèles Claude (facultatif) : il faut un secret ANTHROPIC_API_KEY dans Colab.
# Coût du test : moins de 0,50 $ pour les deux.
CLAUDE_A_COMPARER = ["claude-haiku-5-5", "claude-opus-5-5"]
try:
    import anthropic
    claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    MODELES_A_COMPARER += CLAUDE_A_COMPARER
except Exception:
    print("Pas de clé ANTHROPIC_API_KEY : les modèles Claude ne sont pas testés.")
print("Modèles testés :", MODELES_A_COMPARER)

## 3. Lecture des 20 articles (une seule fois)

In [ ]:
reference = pd.read_csv(f"{DEPOT}/tests/reference_20.csv", dtype=str).fillna("")
passe1 = requests.get(f"{DEPOT}/config/consignes/passe1_v1.txt").text
passe2 = requests.get(f"{DEPOT}/config/consignes/passe2_{VERSION_CONSIGNE}.txt").text
liste_techno = ", ".join(pd.read_csv(f"{DEPOT}/config/technologies.csv")["nom"])
NAVIGATEUR = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/130.0 Safari/537.36", "Accept-Language": "fr,en;q=0.8"}

def lire_article(url):
    brut = None
    try:
        brut = trafilatura.fetch_url(url)
    except Exception:
        pass
    if not brut:
        try:
            r = requests.get(url, headers=NAVIGATEUR, timeout=30)
            brut = r.text if r.ok else None
        except Exception:
            pass
    if not brut:
        return "", ""
    meta = trafilatura.extract_metadata(brut)
    return (meta.title if meta and meta.title else ""), (trafilatura.extract(brut) or "")

textes = {}
for _, a in reference.iterrows():
    titre, texte = lire_article(a["url"])
    textes[a["num"]] = (titre or a["titre_court"], texte)
    print(a["num"], "·", a["titre_court"], "·", len(texte), "caractères lus")
lus = [n for n, (_, t) in textes.items() if len(t) > 200]
print(f"\n{len(lus)} articles lus sur {len(textes)}. Les autres sont exclus du calcul des scores.")

## 4. Classement par chaque modèle, puis comparaison

In [ ]:
def nettoyer(t):
    return re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()

def demander(modele, consigne, max_tokens, json_force=False):
    if modele.startswith("claude"):
        rep = claude.messages.create(model=modele, max_tokens=max_tokens,
                                     messages=[{"role": "user", "content": consigne}])
        return nettoyer("".join(b.text for b in rep.content if b.type == "text"))
    params = dict(model=modele, temperature=0, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": consigne}])
    if json_force:
        try:
            return nettoyer(client.chat.completions.create(response_format={"type": "json_object"}, **params)
                            .choices[0].message.content)
        except Exception:
            pass
    return nettoyer(client.chat.completions.create(**params).choices[0].message.content)

def lire_tri(t):
    m = re.search(r"\b(OUI|NON|DOUTE)\b", (t or "").upper())
    return m.group(1) if m else "ILLISIBLE"

def extraire_json(t):
    d, f = t.find("{"), t.rfind("}")
    try:
        return json.loads(t[d:f + 1]) if d >= 0 and f > d else None
    except json.JSONDecodeError:
        return None

def classer(modele):
    lignes = []
    for _, a in reference.iterrows():
        titre, texte = textes[a["num"]]
        l = {"num": a["num"], "texte_lu": len(texte)}
        t0 = time.time()
        try:
            l["passe1"] = lire_tri(demander(modele, passe1.replace("<<TITRE>>", titre)
                                           .replace("<<EXTRAIT>>", texte[:500]), 500))
            r2 = demander(modele, passe2.replace("<<TECHNOLOGIES>>", liste_techno).replace("<<TITRE>>", titre)
                          .replace("<<LANGUE>>", a["langue"])
                          .replace("<<TEXTE>>", texte[:6000] or "(texte non lisible : classer sur le titre)"),
                          4000, json_force=True)
            f = extraire_json(r2)
            l["json_valide"] = f is not None
            if f:
                axes = f.get("axes") or []
                l.update({
                    "lentille_principale": f.get("lentille_principale") or "",
                    "lentille_secondaire": f.get("lentille_secondaire") or "AUCUNE",
                    "iceberg": f.get("iceberg") or "",
                    "aire_sujet": ";".join(f.get("aire_sujet") or []),
                    "nature": f.get("nature") or "",
                    "axe": axes[0].get("code", "") if axes else "AUCUN",
                    "pole": axes[0].get("pole", "") if axes else "",
                    "precipitation": f.get("precipitation") or "",
                    "rebond": (f.get("rebond_type") or "") if f.get("rebond") == "MENTIONNE" else "NON_MENTIONNE",
                    "resume": f.get("resume", ""),
                })
        except Exception as e:
            l["erreur"] = str(e)[:150]
        l["duree_s"] = round(time.time() - t0, 1)
        lignes.append(l)
        time.sleep(1)
    return pd.DataFrame(lignes)

CHAMPS = ["lentille_principale", "iceberg", "aire_sujet", "nature", "axe", "pole", "precipitation", "rebond"]
AIRES = {"EU","US","CN","JK","IN","ASE","RU","GO","MO","AF","LA","OC","GLO"}

def noter(df):
    for col in ["passe1", "json_valide", "duree_s"]:
        if col not in df:
            df[col] = None
    c = reference.merge(df, on="num", suffixes=("_ref", "_mod"))
    s = {}
    s["tri"] = c.apply(lambda r: r["passe1"] in ("OUI", "DOUTE") if r["pertinent"] == "OUI"
                       else r["passe1"] == "NON", axis=1).mean()
    s["JSON valide"] = c["json_valide"].fillna(False).mean()
    p = c[(c["pertinent"] == "OUI") & (c["num"].isin(lus))]
    for ch in CHAMPS:
        if f"{ch}_mod" not in p:
            s[ch] = 0.0
            continue
        if ch == "aire_sujet":
            ok = [r in str(m).split(";") for r, m in zip(p[f"{ch}_ref"], p[f"{ch}_mod"])]
        else:
            ok = [str(r) == str(m) for r, m in zip(p[f"{ch}_ref"], p[f"{ch}_mod"])]
        s[ch] = sum(ok) / max(len(ok), 1)
    s["codes d'aire inventés"] = sum(any(x and x not in AIRES for x in str(v).split(";"))
                                     for v in c.get("aire_sujet_mod", []))
    s["secondes par article"] = df["duree_s"].mean()
    return s, c

resultats, details = {}, {}
for m in MODELES_A_COMPARER:
    print("Classement par", m, "…")
    try:
        df = classer(m)
        if "erreur" in df and df["erreur"].notna().all():
            print("   ÉCHEC sur tous les articles. Message :", df["erreur"].dropna().iloc[0])
            continue
        resultats[m], details[m] = noter(df)
        n_err = int(df["erreur"].notna().sum()) if "erreur" in df else 0
        print("   terminé." + (f" ({n_err} erreurs, ex. : {df['erreur'].dropna().iloc[0]})" if n_err else ""))
    except Exception as e:
        print("   ÉCHEC :", str(e)[:300])

tableau = pd.DataFrame(resultats)
SEUILS = {"tri": 0.95, "JSON valide": 0.98, "lentille_principale": 0.80, "iceberg": 0.65,
          "aire_sujet": 0.90, "nature": 0.80, "axe": 0.70, "pole": 0.70, "precipitation": 0.70, "rebond": 0.70}
tableau.insert(0, "seuil visé", pd.Series(SEUILS))
print(f"Consigne {VERSION_CONSIGNE} · scores calculés sur les {len(lus)} articles lus")
display(tableau.style.format(lambda v: f"{v:.0%}" if isinstance(v, float) and v <= 1 else f"{v:.1f}" if isinstance(v, float) else v))

## 5. Détail et fichiers à télécharger

Un fichier par modèle : chaque étiquette en deux colonnes, `_ref` (la référence) et `_mod` (le modèle), plus le résumé produit. Utile pour juger la qualité des résumés.

In [ ]:
from google.colab import files
tableau.to_csv("comparaison_modeles.csv")
files.download("comparaison_modeles.csv")
for m, c in details.items():
    nom = f"detail_{m}.csv"
    c.to_csv(nom, index=False)
    files.download(nom)

## Comment lire le tableau

- **Une colonne par modèle**, une ligne par étiquette, et la colonne « seuil visé » (L2, section 6.2).
- Sur 11 articles lus, **un article = 9 points** : ne départagez pas deux modèles à moins de 15 points d'écart.
- Regardez aussi la **durée par article** : à 10 000 articles par mois, elle détermine si le classement tient dans la journée.
- Envoyez le tableau à Claude (copier-coller ou fichier `comparaison_modeles.csv`).